# Verified Training

## What This Is
Verified training incorporates the verification objective into the training loss, producing models that are certifiably robust by construction.

**Interval Bound Propagation (IBP) training**: During training, compute the IBP lower bound on the margin (correct class - max other class) for all inputs in the ε-ball. Minimize a loss that penalizes when this bound is negative. The resulting model is certified robust at the training ε.

**DiffAI**: Uses abstract interpretation as a differentiable layer in the training loop. Trains with zonotope or box domain propagation as part of the forward pass.

**Tradeoff**: IBP-trained models have lower nominal accuracy than standard training because they optimize for a harder objective. The certified accuracy gap narrowed significantly with the introduction of CROWN-IBP (mixing IBP loss with cross-entropy).

In [ ]:
import numpy as np
from typing import Tuple

np.random.seed(42)

def sigmoid(z): return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

# IBP training: optimize for certified robustness
# Loss = standard_ce_loss + kappa * ibp_loss

N, D = 300, 4
X = np.random.randn(N, D)
y = (X[:, 0] + X[:, 1] > 0).astype(float)

X_train, y_train = X[:200], y[:200]
X_test, y_test = X[200:], y[200:]

def ibp_loss(X, y, W1, b1, W2, b2, eps):
    """IBP-based certified robustness loss."""
    losses = []
    for x, yt in zip(X[:50], y[:50]):  # subsample for speed
        xlo, xhi = x - eps, x + eps
        W1p, W1n = np.maximum(0, W1), np.minimum(0, W1)
        pre_lo = W1p @ xlo + W1n @ xhi + b1
        pre_hi = W1p @ xhi + W1n @ xlo + b1
        h_lo, h_hi = np.maximum(0, pre_lo), np.maximum(0, pre_hi)
        W2p, W2n = np.maximum(0, W2[0]), np.minimum(0, W2[0])
        # Worst-case output for positive class
        if yt == 1:
            out_worst = W2p @ h_lo + W2n @ h_hi + b2[0]
        else:
            out_worst = W2p @ h_hi + W2n @ h_lo + b2[0]
        target_sign = 2 * yt - 1
        margin = target_sign * out_worst
        losses.append(np.log(1 + np.exp(-margin)))
    return np.mean(losses)

# Compare standard vs IBP training
def train_standard(lr=0.05, epochs=200):
    W1 = np.random.randn(8, D) * 0.2
    b1 = np.zeros(8)
    W2 = np.random.randn(1, 8) * 0.2
    b2 = np.zeros(1)
    for _ in range(epochs):
        h = np.maximum(0, X_train @ W1.T + b1)
        p = sigmoid((h @ W2.T + b2)[:, 0])
        e = p - y_train
        W1 -= lr * (e[:, None] * p[:, None] * (1-p[:, None]) * (h>0) @ W1.reshape(-1,D) * 0 + 0)  # skip
        # Simplified SGD
        dW2 = (e * p * (1-p))[:, None] * h
        W2 -= lr * dW2.mean(0, keepdims=True)
        dh = np.outer(e * p * (1-p), W2[0]) * (h > 0)
        W1 -= lr * (dh.T @ X_train) / len(X_train)
    return W1, b1, W2, b2

W1s, b1s, W2s, b2s = train_standard()

def cert_acc(X, y, W1, b1, W2, b2, eps):
    certified = 0
    for x, yt in zip(X, y):
        xlo, xhi = x - eps, x + eps
        W1p, W1n = np.maximum(0, W1), np.minimum(0, W1)
        p_lo = W1p@xlo + W1n@xhi + b1
        p_hi = W1p@xhi + W1n@xlo + b1
        h_lo, h_hi = np.maximum(0, p_lo), np.maximum(0, p_hi)
        W2p, W2n = np.maximum(0, W2[0]), np.minimum(0, W2[0])
        out_lo = W2p@h_lo + W2n@h_hi + b2[0]
        out_hi = W2p@h_hi + W2n@h_lo + b2[0]
        if yt == 1 and out_lo > 0: certified += 1
        elif yt == 0 and out_hi < 0: certified += 1
    return certified / len(y)

h_te = np.maximum(0, X_test @ W1s.T + b1s)
nominal_acc = ((sigmoid(h_te @ W2s.T + b2s)[:, 0] > 0.5).astype(int) == y_test).mean()

print('Standard Training Results:')
print(f'  Nominal accuracy: {nominal_acc:.3f}')
for eps in [0.1, 0.2, 0.5]:
    ca = cert_acc(X_test, y_test, W1s, b1s, W2s, b2s, eps)
    print(f'  Certified accuracy at eps={eps}: {ca:.3f}')
print('\nNote: IBP training would raise certified accuracy at the cost of nominal accuracy')
